# 05. Segment and Offer Analysis

## Business question
**Which customer segment should the café target with which offer, and why?**

## Purpose
Compare the 10 offers with each other, then look at how each offer performs for different customer segments (age, income, tenure, gender).


## Metrics
| Metric | Formula | Question it answers |
|---|---|---|
| `view_rate` | viewed / received | Do customers see the offer? |
| `influenced_rate` | completed after view / received | How often does an offer end in a completion after being seen? |
| `viewed_to_influenced` | completed after view / viewed | Of the offers that were seen, how many were completed? |
| `share_completed_without_view` | completed without view / all completed | How much reward is paid to customers who had not seen the offer? |
| `reward_per_influenced` | total reward paid / completed after view | What does each "seen, then completed" offer cost in rewards? |

## How offers are named
`type-difficulty-reward-duration`, for example `discount-7-3-7` = spend $7, get $3 off, valid for 7 days. Informational offers have no spend or reward, so they show as `informational-0-0-3`.

## Important
- **Informational offers** have no completion event, so they are measured differently (purchases after viewing) and are shown in their own table. Their numbers cannot be compared with the completion rates of BOGO and discount offers.
- Results show **association, not cause**. There is no group of customers who received no offer, so we cannot measure how much extra spending an offer creates.

## 1. Load the data

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency   # statistical test used in section 5

funnel = pd.read_csv('../data/cleaned/offer_funnel.csv')
customers = pd.read_csv('../data/cleaned/customers_features.csv')
offers = pd.read_csv('../data/cleaned/offers_clean.csv')

# only the columns we need from the informational analysis (matched to the funnel by received_event_id)
info_out = pd.read_csv('../data/cleaned/informational_offer_analysis.csv',
                       usecols=['received_event_id', 'any_tx_in_window', 'tx_after_view_flag', 'spend_in_window'])

## 2. Give each offer a readable label
The offers table identifies offers by a long random id. We build a label from the offer's own properties, so a chart or table can be read without a lookup:

`offer_label = type - difficulty - reward - duration`

| Label | Channels |
|---|---|
| `bogo-5-5-5` | web, email, mobile, social |
| `bogo-5-5-7` | web, email, mobile |
| `bogo-10-10-5` | web, email, mobile, social |
| `bogo-10-10-7` | email, mobile, social |
| `discount-7-3-7` | web, email, mobile, social |
| `discount-10-2-7` | web, email, mobile |
| `discount-10-2-10` | web, email, mobile, social |
| `discount-20-5-10` | web, email |
| `informational-0-0-3` | email, mobile, social |
| `informational-0-0-4` | web, email, mobile |

The label does not include the channels, so the table above is the key to them. The `assert` checks that no two offers share a label.

In [4]:
# build the label from four columns (numbers converted to text so they can be joined)
offers['offer_label'] = (offers['offer_type'] + '-' + offers['difficulty'].astype(str) + '-'
                         + offers['reward'].astype(str) + '-' + offers['duration'].astype(str))

# two offers with the same label could not be told apart later
assert offers['offer_label'].is_unique

offers[['offer_label', 'offer_id', 'channels']].sort_values('offer_label')

,offer_label,offer_id,channels
1,bogo-10-10-5,4d5c57ea9a6940dd891ad53e9dbe8da0,"['web', 'email', 'mobile', 'social']"
0,bogo-10-10-7,ae264e3637204a6fb9bb56bc8210ddfd,"['email', 'mobile', 'social']"
8,bogo-5-5-5,f19421c1d4aa40978ebb69ca19b0e20d,"['web', 'email', 'mobile', 'social']"
3,bogo-5-5-7,9b98b8c7a33c4b65b9aebfe6a799e6d9,"['web', 'email', 'mobile']"
6,discount-10-2-10,fafdcd668e3743c1bb461111dcafc2a4,"['web', 'email', 'mobile', 'social']"
9,discount-10-2-7,2906b810c7d4411798c6938adc9daaa5,"['web', 'email', 'mobile']"
4,discount-20-5-10,0b1e1539f2cc45b7b9fa7c272da2e1d7,"['web', 'email']"
5,discount-7-3-7,2298d6c36e964ae4a3e7e9706d1fb8c2,"['web', 'email', 'mobile', 'social']"
7,informational-0-0-3,5a8bc65990b245e5a138643cd4eb9837,"['email', 'mobile', 'social']"
2,informational-0-0-4,3f207df678b143eea3cee63160fa8bed,"['web', 'email', 'mobile']"


## 3. Join offers, customer segments and informational results onto the funnel
Each funnel row is one offer received by one customer. We add:
- the customer's segments (gender, age group, income group, tenure group),
- the offer label,
- for informational offers, whether the customer purchased after viewing.

`validate='many_to_one'` makes pandas stop with an error if the right-hand table has repeated keys (for example a customer listed twice), which would silently duplicate funnel rows. The `assert` lines check that no rows were added or lost and that every offer and customer matched.

The last step puts each group into its natural order. A CSV loses this order, and without it `100-120k` would sort before `30-50k`.

In [5]:
seg_cols = ['customer_id', 'gender', 'age_group', 'income_group', 'tenure_group']

df = funnel.merge(customers[seg_cols], on='customer_id', how='left', validate='many_to_one')
df = df.merge(offers[['offer_id', 'offer_label']], on='offer_id', how='left', validate='many_to_one')

# left join: only informational rows get values, all other rows stay empty
df = df.merge(info_out, on='received_event_id', how='left', validate='one_to_one')

# checks: same number of rows as the funnel, and nothing failed to match
assert len(df) == len(funnel)
assert df['offer_label'].notna().all()
assert df[['gender', 'age_group', 'income_group', 'tenure_group']].notna().all().all()

# natural order of offers (by type, then difficulty, reward, duration) and of segments
label_order = offers.sort_values(['offer_type', 'difficulty', 'reward', 'duration'])['offer_label'].tolist()
df['offer_label'] = pd.Categorical(df['offer_label'], categories=label_order, ordered=True)
df['income_group'] = pd.Categorical(df['income_group'], ordered=True,
    categories=['30-50k Lower-Middle', '50-75k Middle', '75-100k Upper-Middle', '100-120k Affluent'])
df['age_group'] = pd.Categorical(df['age_group'], ordered=True,
    categories=['18-24 Young Adults', '25-34 Early Career', '35-44 Young Families',
                '45-54 Mature Professionals', '55-64 Pre-Retirement', '65+ Retirees'])
df['tenure_group'] = pd.Categorical(df['tenure_group'], ordered=True,
    categories=['New (<1yr)', 'Established (1-3yr)', 'Loyal (3-5yr)', 'Veteran (5yr+)'])   # no customer is in Veteran with the current bins
df['gender'] = pd.Categorical(df['gender'], categories=['F', 'M', 'O'], ordered=True)

df.shape

(66501, 24)

## 4. Summary functions
Two functions calculate the metrics for any grouping. `by` is a list of columns, for example `['offer_label']` for one row per offer, or `['income_group', 'offer_label']` for one row per segment and offer.

**`completable_summary`** covers BOGO and discount offers, the ones that can be completed.
- `reward_paid` = the offer's reward for every received offer that was completed (viewed or not). It is an estimate of reward cost.
- Summing a True/False column counts the True values.

**`informational_summary`** covers informational offers using purchase activity instead of completion (see `04_informational_offer`).
- `purchase_after_view` = the customer made a purchase in the offer window, at or after their first view.
- `any_purchase` = the customer made a purchase in the offer window, viewed or not.
- The merge in section 3 left these two columns empty for BOGO and discount rows, which makes them non-boolean. We cast them to True/False after keeping only informational rows.

`observed=True` tells pandas to ignore groups with no rows (such as the empty "Veteran" tenure group).

In [6]:
def completable_summary(data, by):
    d = data[data['offer_type'].ne('informational')].copy()
    d['reward_paid'] = d['reward'] * d['completed_any']       # reward paid on each completed offer

    s = (d.groupby(by, observed=True)
         .agg(received=('viewed', 'size'),
              viewed=('viewed', 'sum'),
              completed_any=('completed_any', 'sum'),
              influenced=('influenced_completion', 'sum'),
              reward_paid=('reward_paid', 'sum'))
         .reset_index())

    s['view_rate'] = s['viewed'] / s['received']
    s['influenced_rate'] = s['influenced'] / s['received']
    s['viewed_to_influenced'] = s['influenced'] / s['viewed']
    s['share_completed_without_view'] = (s['completed_any'] - s['influenced']) / s['completed_any']
    s['reward_per_influenced'] = s['reward_paid'] / s['influenced']
    return s


def informational_summary(data, by):
    d = data[data['offer_type'].eq('informational')].copy()
    d['tx_after_view_flag'] = d['tx_after_view_flag'].astype(bool)
    d['any_tx_in_window'] = d['any_tx_in_window'].astype(bool)

    s = (d.groupby(by, observed=True)
         .agg(received=('viewed', 'size'),
              viewed=('viewed', 'sum'),
              purchase_after_view=('tx_after_view_flag', 'sum'),
              any_purchase=('any_tx_in_window', 'sum'))
         .reset_index())

    s['view_rate'] = s['viewed'] / s['received']
    s['purchase_after_view_rate'] = s['purchase_after_view'] / s['received']
    s['purchase_after_view_of_viewed'] = s['purchase_after_view'] / s['viewed']
    s['any_purchase_rate'] = s['any_purchase'] / s['received']
    return s

## 5. How does each offer perform?
One row per offer. Compare offers with each other, not with informational offers, which are shown separately below.

Things to look at:
- **`view_rate`**: how many customers saw the offer.
- **`influenced_rate` and `viewed_to_influenced`**: how often it ends in a completion after being seen.
- **`share_completed_without_view`**: reward paid to customers who never saw the offer first.
- **`reward_per_influenced`**: reward cost per "seen, then completed" offer. Compare this between offers of the same type only, because BOGO and discount rewards are different kinds of reward.

In [7]:
offer_table = completable_summary(df, ['offer_label'])
offer_table[['offer_label', 'received', 'view_rate', 'influenced_rate', 'viewed_to_influenced',
             'share_completed_without_view', 'reward_per_influenced']].round(3)

,offer_label,received,view_rate,influenced_rate,viewed_to_influenced,share_completed_without_view,reward_per_influenced
0,bogo-5-5-5,6576,0.952,0.510,0.536,0.177,6.073
1,bogo-5-5-7,6685,0.518,0.302,0.583,0.512,10.250
2,bogo-10-10-5,6593,0.951,0.412,0.434,0.171,12.067
3,bogo-10-10-7,6683,0.879,0.382,0.435,0.291,14.111
4,discount-7-3-7,6655,0.957,0.616,0.643,0.153,3.543
5,discount-10-2-7,6631,0.514,0.305,0.593,0.475,3.809
6,discount-10-2-10,6652,0.963,0.646,0.671,0.128,2.294
7,discount-20-5-10,6726,0.328,0.191,0.582,0.608,12.769


### Informational offers
Measured by purchases, not completions. `purchase_after_view_rate` and `any_purchase_rate` use all received offers as the denominator.

In [8]:
info_table = informational_summary(df, ['offer_label'])
info_table[['offer_label', 'received', 'view_rate', 'purchase_after_view_rate',
            'purchase_after_view_of_viewed', 'any_purchase_rate']].round(3)

,offer_label,received,view_rate,purchase_after_view_rate,purchase_after_view_of_viewed,any_purchase_rate
0,informational-0-0-3,6643,0.815,0.496,0.609,0.636
1,informational-0-0-4,6657,0.477,0.294,0.616,0.628


## 6. How does each offer perform for each segment?
For each segment type, one table shows the `influenced_rate` of every offer (rows) in every segment group (columns).

**Small groups are hidden.** A rate based on a few customers is unreliable, so any cell with fewer than `MIN_N` received offers is left blank. The smallest cell in each table is printed so you can judge how much to trust the numbers. As a rough guide, a rate based on about 450 offers can be off by around 4 to 5 percentage points.

The "Other" gender group is small and most of its cells will be blank.

In [9]:
MIN_N = 300   # hide cells with fewer received offers than this

for seg in ['income_group', 'age_group', 'tenure_group', 'gender']:
    s = completable_summary(df, [seg, 'offer_label'])

    # reshape: one row per offer, one column per segment group
    rate = s.pivot(index='offer_label', columns=seg, values='influenced_rate')
    n = s.pivot(index='offer_label', columns=seg, values='received')

    print(f"Influenced rate by offer and {seg}   (smallest cell: {int(n.min().min())} received offers)")
    display(rate.where(n >= MIN_N).round(3))      # .where keeps cells with enough data, blanks the rest

Influenced rate by offer and income_group   (smallest cell: 425 received offers)


income_group,30-50k Lower-Middle,50-75k Middle,75-100k Upper-Middle,100-120k Affluent
offer_label,,,,
bogo-5-5-5,0.397,0.514,0.597,0.647
bogo-5-5-7,0.179,0.333,0.417,0.233
bogo-10-10-5,0.229,0.412,0.585,0.588
bogo-10-10-7,0.259,0.392,0.483,0.467
discount-7-3-7,0.546,0.618,0.683,0.654
discount-10-2-7,0.177,0.344,0.421,0.203
discount-10-2-10,0.568,0.645,0.705,0.762
discount-20-5-10,0.087,0.214,0.300,0.103


Influenced rate by offer and age_group   (smallest cell: 366 received offers)


age_group,18-24 Young Adults,25-34 Early Career,35-44 Young Families,45-54 Mature Professionals,55-64 Pre-Retirement,65+ Retirees
offer_label,,,,,,
bogo-5-5-5,0.396,0.436,0.488,0.523,0.535,0.538
bogo-5-5-7,0.171,0.197,0.330,0.340,0.311,0.315
bogo-10-10-5,0.228,0.259,0.397,0.459,0.436,0.459
bogo-10-10-7,0.276,0.292,0.395,0.396,0.404,0.400
discount-7-3-7,0.553,0.540,0.625,0.632,0.622,0.633
discount-10-2-7,0.166,0.186,0.349,0.324,0.339,0.317
discount-10-2-10,0.527,0.575,0.654,0.656,0.672,0.663
discount-20-5-10,0.111,0.088,0.202,0.218,0.212,0.198


Influenced rate by offer and tenure_group   (smallest cell: 562 received offers)


tenure_group,New (<1yr),Established (1-3yr),Loyal (3-5yr)
offer_label,,,
bogo-5-5-5,0.369,0.693,0.509
bogo-5-5-7,0.227,0.388,0.346
bogo-10-10-5,0.294,0.602,0.240
bogo-10-10-7,0.284,0.550,0.194
discount-7-3-7,0.466,0.786,0.748
discount-10-2-7,0.220,0.401,0.371
discount-10-2-10,0.482,0.832,0.790
discount-20-5-10,0.140,0.259,0.170


Influenced rate by offer and gender   (smallest cell: 72 received offers)


gender,F,M,O
offer_label,,,
bogo-5-5-5,0.573,0.461,NaN
bogo-5-5-7,0.335,0.274,NaN
bogo-10-10-5,0.530,0.325,NaN
bogo-10-10-7,0.468,0.319,NaN
discount-7-3-7,0.664,0.580,NaN
discount-10-2-7,0.332,0.282,NaN
discount-10-2-10,0.702,0.607,NaN
discount-20-5-10,0.215,0.169,NaN


## 7. Did each offer go to a similar mix of customers?
If offers were sent to different kinds of customers, then a difference in results could come from **who** received the offer and not from the offer itself. We test whether each offer's segment mix is the same.

- `p` is the chance of seeing a mix this uneven if all offers went to the same kind of customers. A large p (above 0.05) means no evidence that the mix differs.
- Cramér's V measures how strong the link is between offer and segment: 0 means none, and values close to 0.1 or higher would start to matter.

In [10]:
for seg in ['age_group', 'income_group', 'gender', 'tenure_group']:
    table = pd.crosstab(df['offer_label'], df[seg])
    table = table.loc[:, table.sum() > 0]                     # drop empty groups (e.g. Veteran)
    chi2, p, dof, _ = chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    print(f"{seg}: p = {p:.3f}, Cramér's V = {v:.4f}")

age_group: p = 0.333, Cramér's V = 0.0121
income_group: p = 0.891, Cramér's V = 0.0096
gender: p = 0.898, Cramér's V = 0.0091
tenure_group: p = 0.816, Cramér's V = 0.0097


## 8. Findings, interpretation, assumptions and limitations

### What the data shows
1. **View rate separates the offers by whether they used social media.** The five BOGO and discount offers sent on social were viewed 87.9% to 96.3% of the time. The three not on social were viewed only 32.8% to 51.8% of the time. The informational offers show the same pattern (81.5% vs 47.7%).
2. **Offers that are not seen are still completed, and rewarded.** For offers not on social, 47% to 61% of completions had no view before them. For offers on social it was 13% to 29%.
3. **Best performers:** `discount-10-2-10` (64.6% influenced, $2.29 reward per influenced completion) and `discount-7-3-7` (61.6%, $3.54). **Weakest:** `discount-20-5-10` (19.1% influenced, 60.8% of its completions had no view, $12.77 per influenced completion).
4. **Harder BOGO offers cost more and complete less.** `bogo-10-10-5` pays $12.07 per influenced completion at a 41.2% rate. `bogo-5-5-5` pays $6.07 at 51.0%.
5. **Each offer went to a similar customer mix** (Cramér's V about 0.01 for age, income, gender and tenure; p between 0.33 and 0.90), and each offer was received about 6,600 times. Differences between offers are therefore not explained by who received them on these characteristics.
6. **Income and age:** for every offer, the influenced rate rises from the 30-50k to the 50-75k to the 75-100k income group. The 100-120k group is mixed: it does very well on social offers (`discount-10-2-10`: 76.2%) and poorly on offers without social (`discount-20-5-10`: 10.3%). Customers aged 18 to 34 respond less than older customers on every offer (for example `bogo-10-10-5`: 22.8% for ages 18-24 vs 45.9% for 65+).
7. **Tenure:** members of 1 to 3 years have the highest influenced rate on every offer. Among members of 3 to 5 years, the two BOGO offers with a $10 spend are low (`bogo-10-10-5`: 24.0%, `bogo-10-10-7`: 19.4%), while the discount offers stay high (74.8% to 79.0% for `discount-7-3-7` and `discount-10-2-10`).
8. **Gender:** women have a higher influenced rate than men on every offer (for example `bogo-10-10-5`: 53.0% vs 32.5%). The "Other" group is too small to report by offer.
9. **Informational offers:** the offer on social was viewed far more often (81.5% vs 47.7%), but the share of customers who purchased in the window was almost the same (63.6% vs 62.8%).

### Interpretation (reasonable, but not proven)
- Tenure groups in this data may partly reflect when customers signed up, so the tenure differences should be read with care.
- **Visibility is probably the main driver of results, and social is the likely reason.** Two matched pairs support this. `bogo-5-5-5` and `bogo-5-5-7` have the same spend and reward, and the offer with the longer duration does much worse (51.0% vs 30.2%). The same holds for `discount-10-2-10` and `discount-10-2-7` (64.6% vs 30.5%).
- Reward paid without a prior view suggests some rewards go to customers who would have bought anyway.

### Assumptions
- Reward cost is estimated as the offer's reward for every completed offer.
- A completion "after view" means the customer viewed the offer at or before completing it (see `03_offer_funnel`).
- Only measured customer characteristics (age, income, gender, tenure) were checked for balance between offers.

### Limitations
- **No customers without an offer,** so we know which offers complete best, not how much extra spending they cause.
- **Channel is bundled with the offer.** The offers also differ in duration and difficulty, and there are only 10 of them, so the effect of social cannot be separated from other design differences.
- **Small cells.** The 100-120k income group has about 450 received offers per offer, so its rates are noisy.
- **Informational offers** are measured by purchases, and cannot be compared with completion rates.
- **A 30-day snapshot** of customers with complete profiles only.

